# React → Leptos conversion eval

Convert TSX to Leptos 0.7 with a local Ollama model, verify with `cargo check`, grade fidelity, and compare models.

**Run the cells in order.** Cell 3 warms the cargo cache — slow the first time (a few minutes). Paths (`leptos-check/`, `out/`, `samples/`) are relative to the notebook's working directory.

## Cell 1: Install

Python deps via pip. You also need cargo on your PATH (rustup) — the `!cargo --version` line checks that.

In [1]:
# Ensure cargo is on the kernel PATH even if the notebook server was started
# from a shell that hadn't sourced ~/.cargo/env (e.g. rustup installed later).
import os, pathlib
_cargo_bin = str(pathlib.Path.home() / ".cargo" / "bin")
if (pathlib.Path(_cargo_bin) / "cargo").exists() and _cargo_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = _cargo_bin + os.pathsep + os.environ["PATH"]
del _cargo_bin

%pip install -q gradio openai pandas
!cargo --version


Note: you may need to restart the kernel to use updated packages.
cargo 1.100.0 (3acc79c0c 2026-11-05)


## Cell 2: Config and prompt

Ollama serves an OpenAI-compatible endpoint at `http://localhost:11434/v1`. The system prompt below is `prompts/react_to_leptos_v2.md` from the repo root, embedded so the notebook is self-contained.

In [2]:
import os, re, subprocess, pathlib, json, time
import pandas as pd
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
DEFAULT_MODEL = "gpt-oss:20b"
CRATE = pathlib.Path("leptos-check").resolve()

SYSTEM_PROMPT = """You convert React/TypeScript components to Leptos 0.7 (Rust).

TARGET
- Leptos 0.7 only. Start with `use leptos::prelude::*;`
- There is no `Scope`/`cx` parameter. Do not use `#[derive(Props)].`
- Props are plain parameters on a `#[component]` function.
- Dependencies available: `leptos = "0.7"` and `tailwind_fuse`. Do not use other crates.

HARD RULES
- Use only APIs you are certain exist in Leptos 0.7 and the std library. Never invent types or macro syntax.
- Preserve behaviour and class strings exactly. Do not rename, reorder or "improve" classes.
- Preserve declaration order. Enum variants must appear in the same order as the TypeScript union members.
- Rust has no default parameter values. Use `#[prop(optional)]` or `#[prop(default = expr)].`

MAPPING RULES
- String-literal unions become enums deriving `Debug, Clone, Copy, PartialEq, Eq, Default`.
- `Record<Union, string>` lookup maps become an `impl` method on the enum returning `&'static str`.
- Optional props: `#[prop(optional)] name: T` where `T: Default`.
- String props: `#[prop(optional, into)] class: String`.
- `children?: ReactNode` becomes `children: Children` or `Option<Children>`.
- `useState` -> `signal()`/`RwSignal`, `useMemo` -> `Memo`, `useEffect` -> `Effect`, callbacks -> `Callback<T>`.
- `...props` / `React.HTMLAttributes`: expose specific attributes (`id`, `style`, `aria-*`).
- `cn(...)`: use `tailwind_fuse::tw_merge!` if caller class overrides matter, otherwise plain concatenation.

RENDERING
- Render with `view! { ... }` only. The macros `html!`, `rsx!`, `jsx!` and `yew::html!` do not exist in Leptos 0.7.
- Class attribute: `<div class=classes>`. Children: `{children()}`, or `{children.map(|c| c())}` when `Option<Children>`.
- Compute class strings in a `let` above `view!`, not inline in the macro.

PROPS
- TSX destructuring default (`cols = 2`): use `#[prop(default = 2)]`. Do not also add `optional`.
- TSX `?` with no default: `#[prop(optional)]`.
- `children?: ReactNode` is `#[prop(optional)] children: Option<Children>`. Required `children` is `children: Children`.
- `number` props: `u32`.
- The `#[default]` variant is the one matching the TypeScript default, which may not be the first variant. Keep the variants in source order anyway.
- Enums and components are `pub`.

HELPERS AND SOURCE BUGS
- Lookup tables and helpers are module-level functions, never defined inside a component. Use `match` returning `&'static str`.
- Every binding must be used.

EXAMPLE OF A CORRECT SIGNATURE
```rust
#[component]
pub fn Container(
    #[prop(optional)] max_width: ContainerMaxWidth,
    #[prop(optional)] horizontal_padding: ContainerHorizontalPadding,
    #[prop(optional, into)] class: String,
    children: Children,
) -> impl IntoView {
```

OUTPUT
- A single ```rust code block, then nothing else.
- Comments only for TODOs and the cn() decision.
- If cannot be converted faithfully, reply exactly: NEEDS-HUMAN: <one-line reason>
"""

## Cell 3: Scratch crate and compile check

`leptos-check` is a scratch lib crate the generated code gets written into for `cargo check`.

In [3]:
def bootstrap_crate():
    if not CRATE.exists():
        subprocess.run(["cargo", "new", "--lib", str(CRATE)], check=True)
        subprocess.run(["cargo", "add", "leptos@0.7", "--features", "csr"], cwd=CRATE, check=True)
        subprocess.run(["cargo", "add", "tailwind_fuse"], cwd=CRATE, check=True)
    # warm the cache so the first real check is fast
    (CRATE / "src" / "lib.rs").write_text("")
    r = subprocess.run(["cargo", "check"], cwd=CRATE, capture_output=True, text=True)
    return r.returncode == 0, r.stderr

def compile_check(rust: str, timeout=180):
    (CRATE / "src" / "lib.rs").write_text(rust)
    r = subprocess.run(
        ["cargo", "check", "--message-format=short"],
        cwd=CRATE, capture_output=True, text=True, timeout=timeout,
        env={**os.environ, "RUSTFLAGS": "-D warnings"},
    )
    return r.returncode == 0, r.stderr.strip()

ok, log = bootstrap_crate()   # slow once, a few minutes
print("ready" if ok else log)

ready


## Cell 4: Convert with one compiler-feedback retry

Returns `{"rust", "status", "log", "attempts"}` where status is `compiled`, `failed`, or `needs-human`.

In [4]:
def extract_rust(text: str) -> str:
    m = re.search(r"```rust\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def call_model(model, messages):
    r = client.chat.completions.create(model=model, messages=messages, temperature=0.1)
    return r.choices[0].message.content or ""

def convert(tsx: str, model=DEFAULT_MODEL, retries=1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": tsx},
    ]
    attempts = []
    for i in range(retries + 1):
        raw = call_model(model, messages)
        if raw.strip().startswith("NEEDS-HUMAN"):
            return {"rust": "", "status": "needs-human", "log": raw.strip(), "attempts": i + 1}
        rust = extract_rust(raw)
        bad = [m for m in ("html!", "rsx!", "jsx!") if m in rust]
        if bad:
            ok, log = False, f"forbidden macro: {bad}"
        else:
            ok, log = compile_check(rust)
        attempts.append((rust, ok, log))
        if ok:
            return {"rust": rust, "status": "compiled", "log": "", "attempts": i + 1}
        messages += [
            {"role": "assistant", "content": raw},
            {"role": "user", "content": f"That failed `cargo check`:\n{log[:3000]}\nReturn the corrected component in a single rust block."},
        ]
    rust, _, log = attempts[-1]
    return {"rust": rust, "status": "failed", "log": log, "attempts": retries + 1}

## Cell 5: Static fidelity checks

Compiling doesn't prove fidelity, so these cover what `cargo check` can't see.

In [5]:
def fidelity_checks(tsx: str, rust: str) -> dict:
    c = {}
    c["uses_view_macro"] = "view!" in rust
    c["no_props_type"] = not re.search(r"\b(enum|struct)\s+\w*Props\b", rust)
    c["no_panics"] = not re.search(r"unreachable!|panic!|\.unwrap\(\)", rust)
    c["no_bad_import"] = not re.search(r"use\s+[\w:]+!\s*;", rust)
    c["helpers_module_level"] = True  # refine later if needed
    if re.search(r"children\?", tsx):
        c["optional_children"] = "Option<Children>" in rust
    if re.search(r"=\s*[\w'\"\d]+\s*[,}]", tsx):  # destructured defaults
        c["has_prop_default_or_optional"] = "#[prop(" in rust
    if "${" in tsx:  # dynamic tailwind classes in source
        c["dynamic_class_todo"] = "TODO" in rust
    return c

## Cell 6: Gradio UI

Verification harness — another agent produces the Leptos conversion. Paste the generated
Rust here and it runs `cargo check` plus the static fidelity checks (the TSX-conditional
checks are skipped without the React source). `share=False`; `server_name` defaults to localhost.


In [6]:
import gradio as gr

# Some gradio versions (e.g. 6.x) have no "rust" mode in gr.Code; fall back to plain text.
_RUST_LANG = "rust" if "rust" in gr.Code.languages else None

def ui_check(rust: str):
    if not rust.strip():
        return "paste a component first", {}
    ok, log = compile_check(rust)
    checks = fidelity_checks("", rust)  # no TSX source: source-conditional checks are skipped
    status = "compiled" if ok else "failed"
    return status + ("\n\n" + log if log else ""), checks

with gr.Blocks(title="Leptos checker") as demo:
    rust_in = gr.Code(language=_RUST_LANG, label="Leptos output", lines=25)
    status = gr.Textbox(label="Compile status / compiler output", lines=8)
    checks = gr.JSON(label="Fidelity checks")
    gr.Button("Check").click(ui_check, rust_in, [status, checks])

demo.launch(inline=True)   # use share=False; server_name defaults to localhost


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Cell 7: Batch run, grade, compare models

Put your .tsx samples in `samples/`. Outputs land in `out/<model>/`, and each run writes a CSV so you can swap models and compare.

In [7]:
def run_batch(model, samples_dir="samples", retries=1):
    rows = []
    out_dir = pathlib.Path("out") / model.replace(":", "_")
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in sorted(pathlib.Path(samples_dir).glob("**/*.tsx")):
        tsx = f.read_text()
        t0 = time.time()
        res = convert(tsx, model=model, retries=retries)
        checks = fidelity_checks(tsx, res["rust"]) if res["rust"] else {}
        (out_dir / f"{f.stem}.rs").write_text(res["rust"])
        rows.append({
            "file": f.name, "model": model, "status": res["status"],
            "attempts": res["attempts"], "seconds": round(time.time() - t0, 1),
            "checks_passed": sum(checks.values()), "checks_total": len(checks),
            "first_error": res["log"].splitlines()[0] if res["log"] else "",
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "results.csv", index=False)
    return df

df = run_batch(DEFAULT_MODEL)
df

""


In [8]:
models = ["gpt-oss:20b", "qwen2.5-coder:14b"]
all_df = pd.concat([run_batch(m) for m in models])
all_df.pivot(index="file", columns="model", values="status")

KeyError: "The following 'columns' labels are not columns of the DataFrame: ['model']"